In [1]:
# Langkah 1 - Load Data
import numpy as np
import pandas as pd
import os

data_path = 'spam.csv' if os.path.exists('spam.csv') else ('Jobsheet9/spam.csv' if os.path.exists('Jobsheet9/spam.csv') else 'spam.csv')
df = pd.read_csv(data_path, encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [2]:
# Langkah 2 - Preprocessing (Drop kolom, rename, inspeksi, dan encoding)
# Drop kolom tidak berguna
df = df.drop(df.iloc[:, 2:], axis=1)
df.columns = ['Labels', 'SMS']

print("Jumlah Data Per Kelas:")
print(df['Labels'].value_counts())
print("\nInfo Data:")
df.info()

# Encode Label: spam = 1, ham = 0
new_labels = {'spam': 1, 'ham': 0}
df['Labels'] = df['Labels'].map(new_labels)
df.head()

Jumlah Data Per Kelas:
Labels
ham     4825
spam     747
Name: count, dtype: int64

Info Data:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Labels  5572 non-null   str  
 1   SMS     5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
# Langkah 3 - Ekstraksi Fitur dengan CountVectorizer (Bag of Words)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

X = df['SMS'].values
y = df['Labels'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

bow = CountVectorizer()
X_train = bow.fit_transform(X_train)
X_test = bow.transform(X_test)

feat_names = bow.get_feature_names_out() if hasattr(bow, 'get_feature_names_out') else bow.get_feature_names()
print(f"Jumlah kosakata fitur unik: {len(feat_names)}")
print(f"Dimensi data X_train: {X_train.shape}")

Jumlah kosakata fitur unik: 7727
Dimensi data X_train: (4457, 7727)


In [4]:
# Langkah 4 - Training dan Evaluasi Model MultinomialNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

mnb = MultinomialNB()
mnb.fit(X_train, y_train)

y_pred_train = mnb.predict(X_train)
acc_train = accuracy_score(y_train, y_pred_train)

y_pred_test = mnb.predict(X_test)
acc_test = accuracy_score(y_test, y_pred_test)

print(f'Hasil akurasi data train: {acc_train:.4f}')
print(f'Hasil akurasi data test : {acc_test:.4f}')
print("\nConfusion Matrix Data Test:\n", confusion_matrix(y_test, y_pred_test))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred_test, target_names=['Ham', 'Spam']))

Hasil akurasi data train: 0.9946
Hasil akurasi data test : 0.9776

Confusion Matrix Data Test:
 [[951   3]
 [ 22 139]]

Laporan Klasifikasi:
               precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       954
        Spam       0.98      0.86      0.92       161

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115

